## `add_conditional_edges`
add_conditional_edges 是 LangGraph 中用于实现动态路由（conditional branching）的最核心方法之一。它允许在执行完一个节点（source node）后，根据当前状态（state）动态决定下一个节点（可以是一个或多个），这特别适合构建 Agent、决策流程、工具调用循环等场景。

### `add_conditional_edges函数签名`

```text
graph.add_conditional_edges(
    source: str,                                      # 起点节点
    path: Callable | Runnable,                        # 路由决策函数（或 Runnable）
    path_map: dict[Hashable, str] | list[str] | None = None  # 可选的映射
) -> Self
```
`source`: 条件判断的起始节点，执行完该节点后调用path函数  

`path`: 路由函数，接收当前State，返回一个值（或多个值的序列），用于决定下一个节点  

`path_map`: 可选的映射字典（或节点列表），把`path`返回的值翻译成图中注册的`节点名`

### 实验1：基础条件路由
### path + path_map

In [5]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal

In [9]:
class State(TypedDict):
    number: int
    result: str

def classify_number(state:State)->Literal['even', 'odd']:
    print('进入路由函数')
    if state['number'] % 2 == 0:
        return 'even'
    return 'odd'

def handle_even(state: State):
    print('进入even节点')
    return {"result": f"{state['number']} 是偶数"}

def handle_odd(state: State):
    print('进入odd节点')
    return {"result": f"{state['number']} 是奇数"}

graph = StateGraph(State)
graph.add_node("classify", lambda s: s)  # 占位
graph.add_node("even", handle_even)
graph.add_node("odd", handle_odd)

graph.add_edge(START, "classify")
graph.add_conditional_edges(
    "classify",
    classify_number,
    {"even": "even", "odd": "odd"}   # path_map
)
graph.add_edge("even", END)
graph.add_edge("odd", END)

app = graph.compile()

# 测试
print(app.invoke({"number": 42}))
print('='*20)
print(app.invoke({"number": 7}))

进入路由函数
进入even节点
{'number': 42, 'result': '42 是偶数'}
进入路由函数
进入odd节点
{'number': 7, 'result': '7 是奇数'}


### 实验2：模拟ReAct Agent 循环
### LLM → 判断是否继续 → Tools

In [26]:
# 打印state的内容
def debug_state(node_name: str, state):

    print("\n" + "=" * 50)

    print(f"进入节点: {node_name}")

    print("当前 State:")

    for k, v in state.items():
        print(f"  {k}: {v}")

    print("=" * 50)

In [27]:
# 1. 定义State
from typing_extensions import TypedDict

class State(TypedDict):
    question: str
    thought: str
    tool_result: str
    final_answer: str
    need_tool: bool
    step_count: int

In [28]:
# 2. 模拟LLM

def agent_node(state:State):
    print("\n进入 Agent 节点")
    debug_state('agent_node',  state)
    
    step = state.get("step_count", 0)
    # 第一次：决定调用工具
    if step == 0:
        return {
            "thought": "我需要使用计算器工具",
            "need_tool": True,
            "step_count": 1
        }
    # 第二次：根据工具结果回答
    return {
        "final_answer": f"最终答案是：{state['tool_result']}",
        "need_tool": False
    }


from langgraph.graph import START, END, StateGraph
from langchain.messages import AIMessage, HumanMessage

In [29]:
# 3. 定义Tool节点
def calculator_tool(state: State):
    print("\n进入 Tool 节点")
    debug_state('calculator_tool', state)
    question = state["question"]
    # 超简陋计算器
    if "2+2" in question:
        return {
            "tool_result": "4"
        }
    return {
        "tool_result": "未知"
    }

In [30]:
# 4. 定义路由函数
from typing import Literal
def router(state: State)->Literal['tool', '__end__']:
    print("\n进入 Router")
    debug_state('router', state)
    if state["need_tool"]:
        print("Router决定：进入tool")
        return "tool"

    print("Router决定：结束")
    return "__end__"

In [31]:
# 5. 构建Graph
from langgraph.graph import START, END, StateGraph
graph = StateGraph(State)
graph.add_node('agent', agent_node)
graph.add_node('tool', calculator_tool)

graph.add_edge(START, 'agent')

# 条件循环
graph.add_conditional_edges(
    'agent',
    router,
    {
        'tool':'tool',
        '__end__':END
    }
)

graph.add_edge('tool', 'agent')

In [32]:
app = graph.compile()

result = app.invoke({
    "question": "请帮我计算2+2"
})

print("\n最终State：")
print(result)


进入 Agent 节点

进入节点: agent_node
当前 State:
  question: 请帮我计算2+2

进入 Router

进入节点: router
当前 State:
  question: 请帮我计算2+2
  thought: 我需要使用计算器工具
  need_tool: True
  step_count: 1
Router决定：进入tool

进入 Tool 节点

进入节点: calculator_tool
当前 State:
  question: 请帮我计算2+2
  thought: 我需要使用计算器工具
  need_tool: True
  step_count: 1

进入 Agent 节点

进入节点: agent_node
当前 State:
  question: 请帮我计算2+2
  thought: 我需要使用计算器工具
  tool_result: 4
  need_tool: True
  step_count: 1

进入 Router

进入节点: router
当前 State:
  question: 请帮我计算2+2
  thought: 我需要使用计算器工具
  tool_result: 4
  final_answer: 最终答案是：4
  need_tool: False
  step_count: 1
Router决定：结束

最终State：
{'question': '请帮我计算2+2', 'thought': '我需要使用计算器工具', 'tool_result': '4', 'final_answer': '最终答案是：4', 'need_tool': False, 'step_count': 1}


### 实验3: Fan-out并行分支（返回列表）
### 理解一个节点同时触发多个后续节点（并行执行）

In [38]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from operator import add

class State(TypedDict):
    query: str
    results: Annotated[list[str], add]  # Reducer 自动合并列表

def researcher(state: State):
    debug_state('researcher', state)
    return {"results": [f"网页搜索结果 for {state['query']}"]}

def coder(state: State):
    debug_state('coder', state)
    return {"results": [f"代码分析结果 for {state['query']}"]}

def router(state: State):
    # 返回多个节点 → 并行执行
    return ["researcher", "coder"]

graph = StateGraph(State)
graph.add_node("router_node", lambda s: s)
graph.add_node("researcher", researcher)
graph.add_node("coder", coder)
graph.add_node("synthesizer", lambda s: {"results": ["综合结论"]})

graph.add_edge(START, "router_node")
graph.add_conditional_edges("router_node", router)  # 无 path_map，直接返回列表
graph.add_edge("researcher", "synthesizer")
graph.add_edge("coder", "synthesizer")
graph.add_edge("synthesizer", END)

app = graph.compile()
print(app.invoke({"query": "AI 发展趋势"}))


进入节点: coder
当前 State:
  query: AI 发展趋势
  results: []

进入节点: researcher
当前 State:
  query: AI 发展趋势
  results: []
{'query': 'AI 发展趋势', 'results': ['代码分析结果 for AI 发展趋势', '网页搜索结果 for AI 发展趋势', '综合结论']}


### 实验4：动态多分支+Send
### 根据运行时数据动态创建任意数量的分支（最强大用法）

In [42]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from typing import TypedDict, Annotated
from operator import add

class State(TypedDict):
    tasks: list[dict]
    results: Annotated[list[str], add]   # 关键：使用 Reducer 合并多个 worker 的输出

# ==================== 节点函数 ====================
def planner(state: State):
    """Planner 只做规划，返回正常的 State 更新"""
    print("📋 Planner 执行：正在生成任务分支...")
    # 这里可以做真实规划逻辑（比如调用 LLM 生成任务）
    return {"tasks": state["tasks"]}   # 返回 dict（正常节点行为）

def worker(state: dict):
    """每个 worker 独立执行"""
    task_id = state.get("task_id", 0)
    task_name = state.get("task", "未知任务")
    result = f"✅ 完成任务 {task_id}: {task_name}"
    print(result)
    return {"results": [result]}   # 必须返回 dict

def aggregator(state: State):
    """所有 worker 完成后汇总"""
    print("\n🔄 Aggregator 执行：所有任务已完成！")
    return {"results": ["🎉 全部任务处理完毕！"]}

# ==================== 路由函数（关键） ====================
def assign_workers(state: State):
    """专门用于返回 Send 列表的路由函数"""
    print(f"   → 准备启动 {len(state['tasks'])} 个并行 worker...")
    return [
        Send("worker", {"task": t["name"], "task_id": i})
        for i, t in enumerate(state["tasks"])
    ]

# ==================== 构建图 ====================
graph = StateGraph(State)

graph.add_node("planner", planner)
graph.add_node("worker", worker)
graph.add_node("aggregator", aggregator)

graph.add_edge(START, "planner")

# 核心写法：planner 执行完后，走条件路由生成 Send
graph.add_conditional_edges("planner", assign_workers)

graph.add_edge("worker", "aggregator")
graph.add_edge("aggregator", END)

app = graph.compile()

# ==================== 测试运行 ====================
tasks = [
    {"name": "任务1：数据分析"},
    {"name": "任务2：生成报告"},
    {"name": "任务3：发送邮件"}
]

result = app.invoke({"tasks": tasks, "results": []})

print("\n=== 最终 State ===")
print(result)

📋 Planner 执行：正在生成任务分支...
   → 准备启动 3 个并行 worker...
✅ 完成任务 0: 任务1：数据分析
✅ 完成任务 1: 任务2：生成报告
✅ 完成任务 2: 任务3：发送邮件

🔄 Aggregator 执行：所有任务已完成！

=== 最终 State ===
{'tasks': [{'name': '任务1：数据分析'}, {'name': '任务2：生成报告'}, {'name': '任务3：发送邮件'}], 'results': ['✅ 完成任务 0: 任务1：数据分析', '✅ 完成任务 1: 任务2：生成报告', '✅ 完成任务 2: 任务3：发送邮件', '🎉 全部任务处理完毕！']}
